# Шаг 1. Выбор классов и анализ датасета

**Цель:** 
1. Выбрать 12 классов (в пределах требуемых 10–15).
2. Изучить метаданные (размеры картинок, координаты знаков).
3. Подготовить логику группировки по "трекам" (сериям кадров), чтобы избежать утечки данных (data leakage) при разделении на train/val.

In [ ]:
import os
import glob
import pandas as pd
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Укажи путь к папке с изображениями (проверь, куда ты распаковал архив)
BASE_PATH = 'data/GTSRB/Final_Training/Images'

# Выбираем 12 классов (от 0 до 14, пропуская некоторые для разнообразия)
# 0:20 км/ч, 1:30 км/ч, 2:50 км/ч, 3:60 км/ч, 4:70 км/ч, 5:80 км/ч, 
# 7:100 км/ч, 8:120 км/ч, 9:Обгон запрещен, 12:Приоритет, 13:Уступи дорогу, 14:STOP
SELECTED_CLASSES = [0, 1, 2, 3, 4, 5, 7, 8, 9, 12, 13, 14]

CLASS_NAMES = {
    0: 'Speed limit 20', 1: 'Speed limit 30', 2: 'Speed limit 50', 
    3: 'Speed limit 60', 4: 'Speed limit 70', 5: 'Speed limit 80',
    7: 'Speed limit 100', 8: 'Speed limit 120', 9: 'No passing',
    12: 'Priority road', 13: 'Yield', 14: 'Stop'
}

### Анализ метаданных и размеров

In [ ]:
all_dfs = []
image_sizes = []

for cls in SELECTED_CLASSES:
    cls_folder = os.path.join(BASE_PATH, f'{cls:05d}')
    csv_path = os.path.join(cls_folder, f'GT-{cls:05d}.csv')
    
    # Читаем CSV с разметкой
    df = pd.read_csv(csv_path, sep=';')
    df['ClassId'] = cls
    
    # ВАЖНО: Извлекаем TrackID из имени файла ('00001_00015.ppm'префикс до нижнего подчеркивания'00001' - трек)
    df['TrackID'] = df['Filename'].apply(lambda x: x.split('_')[0])
    all_dfs.append(df)
    
    # Собираем информацию о размерах (для первых 50 картинок каждого класса)
    for img_name in df['Filename'][:50]:
        img_path = os.path.join(cls_folder, img_name)
        with Image.open(img_path) as img:
            image_sizes.append(img.size)

# Объединяем все выбранные классы в один датафрейм
df_selected = pd.concat(all_dfs, ignore_index=True)

print(f"Всего изображений в выбранных классах: {len(df_selected)}")
print(f"Уникальных треков (серий): {df_selected['TrackID'].nunique()}")
print("\nПример метаданных (первые 5 строк):")
display(df_selected.head())

# Анализ размеров
widths = [s[0] for s in image_sizes]
heights = [s[1] for s in image_sizes]

print(f"\nРазмеры изображений:")
print(f"Ширина: от {min(widths)} до {max(widths)} пикселей (в среднем {int(np.mean(widths))}, мода {stats.mode(widths)}, медиана {int(np.median(widths))})")
print(f"Высота: от {min(heights)} до {max(heights)} пикселей (в среднем {int(np.mean(heights))}, мода {stats.mode(heights)}, медиана {int(np.median(heights))})")

### Визуализация выбранных классов

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()

for i, cls in enumerate(SELECTED_CLASSES):
    # Берем случайную картинку из класса
    cls_df = df_selected[df_selected['ClassId'] == cls]
    sample_row = cls_df.sample(1).iloc[0]
    
    img_path = os.path.join(BASE_PATH, f'{cls:05d}', sample_row['Filename'])
    img = Image.open(img_path)
    
    # Обрезаем по ROI (Region of Interest), как указано в CSV
    roi = (sample_row['Roi.X1'], sample_row['Roi.Y1'], sample_row['Roi.X2'], sample_row['Roi.Y2'])
    img_cropped = img.crop(roi)
    
    axes[i].imshow(img_cropped)
    axes[i].set_title(f"{cls}: {CLASS_NAMES[cls]}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Шаг 2. Разделение данных без утечки (Group K-Fold / Group Split)

**Почему нельзя разделять случайно?**
В датасете GTSRB изображения сгруппированы в «треки» (Tracks) — это серии последовательных кадров одного и того же физического знака. Если случайно разделить картинки, один и тот же знак попадет и в Train, и в Validation. Модель просто запомнит его, и мы получим ложно завышенную точность.

**Что делаем:**
1. Собираем список всех уникальных `TrackID` (у нас их 75).
2. Делим **треки** на Train и Validation в пропорции 80/20.
3. Используем стратификацию по классам (`ClassId`), чтобы в обеих выборках были представлены все 12 классов.

In [ ]:
from sklearn.model_selection import train_test_split

#  Делаем TrackID уникальным, добавляя к нему номер класса
# Теперь трек 00000 из класса 0 будет называться "0_00000", а из класса 1 — "1_00000"
df_selected['UniqueTrackID'] = df_selected['ClassId'].astype(str) + '_' + df_selected['Filename'].apply(lambda x: x.split('_')[0])

# 2. Группируем по новому уникальному ID
tracks_df = df_selected.groupby('UniqueTrackID')['ClassId'].first().reset_index()

print(f"Всего уникальных треков (с учетом классов): {len(tracks_df)}")

# 3. Теперь стратификация сработает, так как в каждом классе точно больше 1 трека
train_tracks, val_tracks = train_test_split(
    tracks_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=tracks_df['ClassId']
)

print(f"Треков в Train: {len(train_tracks)}")
print(f"Треков в Validation: {len(val_tracks)}")

In [ ]:
# Отбираем картинки по уникальным ID
train_df = df_selected[df_selected['UniqueTrackID'].isin(train_tracks['UniqueTrackID'])].copy()
val_df = df_selected[df_selected['UniqueTrackID'].isin(val_tracks['UniqueTrackID'])].copy()

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Картинок в Train: {len(train_df)}")
print(f"Картинок в Validation: {len(val_df)}")

In [ ]:
# Проверка на утечку (Data Leakage)
train_track_ids = set(train_df['UniqueTrackID'])
val_track_ids = set(val_df['UniqueTrackID'])
intersection = train_track_ids.intersection(val_track_ids)

if len(intersection) == 0:
    print("\n✅ Утечка данных исключена: ни одна серия (трек) не встречается в обеих выборках.")
else:
    print(f"\n❌ ОШИБКА: Найдено {len(intersection)} общих треков!")

# Проверка классов
print("\nРаспределение классов в Train:")
print(train_df['ClassId'].value_counts().sort_index().to_dict())

print("\nРаспределение классов в Validation:")
print(val_df['ClassId'].value_counts().sort_index().to_dict())

### Визуализация (опционально)

In [ ]:
# Найдем один TrackID из валидации, чтобы показать серию
sample_track = val_df['TrackID'].iloc[0]
sample_images = val_df[val_df['TrackID'] == sample_track].head(4)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, (_, row) in enumerate(sample_images.iterrows()):
    cls = row['ClassId']
    img_path = os.path.join(BASE_PATH, f'{cls:05d}', row['Filename'])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"Track: {sample_track}\nFile: {row['Filename']}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# Шаг 3: Пайплайн данных, нормализация и аугментации.
### Трансформации и Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image

# 1. Определяем трансформации
# Для тренировки (с аугментациями)
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)), # Приводим к одному размеру (начнем с 64x64 для скорости)
    transforms.RandomAffine(degrees=10, scale=(0.8, 1.2)), # Аугментация масштаба и небольших поворотов
    transforms.ColorJitter(brightness=0.4, contrast=0.4),  # Аугментация яркости (борьба с темными знаками)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Нормализация ImageNet
])

# Для валидации (ТОЛЬКО ресайз и нормализация, никаких искажений!)
val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Создаем кастомный Dataset
class GTSRBDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        """
        df: DataFrame с колонками 'Filename' и 'ClassId'
        root_dir: путь к папке GTSRB/Final_Training/Images
        transform: наши аугментации
        """
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        
        # Создаем маппинг: наши 12 классов -> от 0 до 11 (для нейросети)
        self.classes = sorted(df['ClassId'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Путь к картинке
        cls_folder = f"{row['ClassId']:05d}"
        img_path = os.path.join(self.root_dir, cls_folder, row['Filename'])
        
        # Открываем картинку (в GTSRB они в формате .ppm, PIL отлично с ними справляется)
        image = Image.open(img_path).convert('RGB')
        
        # Применяем трансформации
        if self.transform:
            image = self.transform(image)
            
        # Получаем метку (сжимаем от 0..14 до 0..11)
        label = self.class_to_idx[row['ClassId']]
        
        return image, label
        

### Создание DataLoader'ов

In [ ]:
# Параметры
BATCH_SIZE = 32

# Создаем датасеты
train_dataset = GTSRBDataset(df=train_df, root_dir=BASE_PATH, transform=train_transforms)
val_dataset = GTSRBDataset(df=val_df, root_dir=BASE_PATH, transform=val_transforms)

# Создаем DataLoader'ы
# DataLoader автоматически перемешивает данные и собирает их в батчи
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Классы для обучения: {train_dataset.classes}")
print(f"Количество батчей в Train: {len(train_loader)}")
print(f"Количество батчей в Validation: {len(val_loader)}")

### Визуализация аугментаций (Проверка)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Функция для "разнормализации" картинки (чтобы вернуть цвета для отображения)
def unnormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.numpy().transpose((1, 2, 0))
    img = std * img + mean
    return np.clip(img, 0, 1)

# Возьмем один пример из датасета
img_tensor, label = train_dataset[0]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    # Каждый раз применяются новые случайные аугментации
    aug_img, _ = train_dataset[0] 
    axes[i].imshow(unnormalize(aug_img))
    axes[i].axis('off')
    if i == 0:
        axes[i].set_title(f"Класс: {label}\nОригинал (с ресайзом)")
    else:
        axes[i].set_title(f"Аугментация {i}")
plt.show()

# Шаг 4. Сборка Baseline CNN и ResNet
### Настройка устройства (CPU/GPU) и метрик

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
import time

# Проверяем, есть ли видеокарта (CUDA). Если нет — работаем на процессоре (CPU).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Обучение будет проходить на: {device}")

# Функция для оценки модели на Validation
def evaluate_model(model, dataloader, criterion):
    model.eval() # Переводим модель в режим проверки
    all_preds = []
    all_labels = []
    running_loss = 0.0
    
    with torch.no_grad(): # Отключаем расчет градиентов (экономит память)
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            # Получаем предсказанные классы
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = running_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro') # Macro F1 важен из-за дисбаланса классов
    
    return val_loss, acc, f1

### Создание Baseline CNN (с нуля): 3 блока свертки (Conv2d + ReLU + MaxPool) и 2 полносвязных слоя.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=12):
        super(BaselineCNN, self).__init__()
        
        # Блок 1 (вход: 3 канала RGB, 64x64)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Блок 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Блок 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        # Полносвязные слои
        # После трех пулингов (64 -> 32 -> 16 -> 8), 
        # размер карты будет 128 x 8 x 8
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.dropout = nn.Dropout(0.5) # Защита от переобучения
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        
        x = x.view(-1, 128 * 8 * 8) # Разворачиваем в вектор
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Создаем модель
baseline_model = BaselineCNN(num_classes=12).to(device)
print("Baseline CNN успешно создана!")

In [ ]:
# универсальная функция для обучения
from tqdm.notebook import tqdm # Импортируем прогресс-бар для Jupyter

def train_model(model, train_loader, val_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    
    for epoch in range(epochs):
        model.train() # Режим обучения
        running_loss = 0.0
        start_time = time.time()
        
        loop = tqdm(train_loader, leave=True, desc=f"Эпоха {epoch+1}/{epochs}")
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Обнуляем градиенты
            optimizer.zero_grad()
            # Прямой проход (Forward)
            outputs = model(images)
            # Вычисление ошибки (Loss)
            loss = criterion(outputs, labels)
            # Обратный проход (Backward)
            loss.backward()
            # Шаг оптимизатора (обновление весов)
            optimizer.step()
            
            running_loss += loss.item()
            loop.set_postfix(loss=running_loss/len(loop))
            
        train_loss = running_loss / len(train_loader)
        
        # Оценка на Validation
        val_loss, val_acc, val_f1 = evaluate_model(model, val_loader, criterion)
        
        # Сохраняем историю
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        print(f"Эпоха {epoch+1}/{epochs} | Время: {time.time()-start_time:.1f}с | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")
        
    return history

In [ ]:
# Обучаем нашу простую модель 5 эпох (для начала)
print("ОБУЧЕНИЕ BASELINE CNN")
baseline_history = train_model(baseline_model, train_loader, val_loader, epochs=5, lr=0.001)

###  Fine-tuning ResNet (Дообучение)
Теперь берем мощную предобученную сеть (ResNet18) и меняем последний слой под наши 12 классов.

In [ ]:
# 1. Трансформации для ResNet (224x224)
train_transforms_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=10, scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms_resnet = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Создаем новые Dataset и DataLoader
train_dataset_resnet = GTSRBDataset(df=train_df, root_dir=BASE_PATH, transform=train_transforms_resnet)
val_dataset_resnet = GTSRBDataset(df=val_df, root_dir=BASE_PATH, transform=val_transforms_resnet)

train_loader_resnet = DataLoader(train_dataset_resnet, batch_size=32, shuffle=True)
val_loader_resnet = DataLoader(val_dataset_resnet, batch_size=32, shuffle=False)

print("DataLoader для ResNet (224x224) готов!")

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

# Загружаем ResNet заново
resnet_model_v2 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# НЕ замораживаем слои! Даем модели полностью адаптироваться (Fine-tuning)
for param in resnet_model_v2.parameters():
    param.requires_grad = True

# Меняем последний слой
num_ftrs = resnet_model_v2.fc.in_features
resnet_model_v2.fc = nn.Linear(num_ftrs, 12)
resnet_model_v2 = resnet_model_v2.to(device)

# Внимание: Learning Rate теперь ОЧЕНЬ маленький (1e-5 или 0.00001), 
# чтобы не сломать то, чему сеть уже научилась на ImageNet
print("ОБУЧЕНИЕ RESNET18 (Полный Fine-tuning)")
# Для экономии времени ставим 3 эпохи (если процессор медленный, 224х224 будет считаться дольше)
resnet_history_v2 = train_model(resnet_model_v2, train_loader_resnet, val_loader_resnet, epochs=3, lr=0.00001)

In [ ]:
import torchvision.models as models

print("ОБУЧЕНИЕ RESNET18 (FINE-TUNING)")

# Загружаем предобученную ResNet18
resnet_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Замораживаем все слои (чтобы не разрушить их веса при первом обучении)
for param in resnet_model.parameters():
    param.requires_grad = False

# Заменяем последний полносвязный слой (он был на 1000 классов ImageNet) на наш (12 классов)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 12)

# Отправляем на устройство
resnet_model = resnet_model.to(device)

# Обучаем. Так как мы заморозили слои, можно ставить learning rate побольше
resnet_history = train_model(resnet_model, train_loader, val_loader, epochs=5, lr=0.001)

# Шаг 5: Матрица ошибок и Анализ ошибок.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Выбираем лучшую модель для анализа (если ResNet не догнала Baseline, берем Baseline)
best_model = baseline_model # или resnet_model_v2, смотря кто победит
best_model.eval()

all_preds = []
all_labels = []

# Прогоняем валидацию
with torch.no_grad():
    for images, labels in val_loader: # Используем DataLoader Baseline (64x64)
        images = images.to(device)
        outputs = best_model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Строим матрицу
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[CLASS_NAMES[c] for c in train_dataset.classes])

fig, ax = plt.subplots(figsize=(12, 10))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
plt.title("Матрица ошибок (Validation)")
plt.show()

In [ ]:
# тест
import torchvision.transforms.functional as TF

def test_dark_image(model, df, idx, darkness_factor=0.2):
    """Функция для теста одной темной картинки"""
    model.eval()
    row = df.iloc[idx]
    
    # Загружаем оригинал
    img_path = os.path.join(BASE_PATH, f"{row['ClassId']:05d}", row['Filename'])
    img_orig = Image.open(img_path).convert('RGB')
    
    # Применяем валидационные трансформации
    img_tensor = val_transforms(img_orig).unsqueeze(0).to(device)
    
    # Искусственно затемняем тензор (умножаем на 0.2)
    img_dark = img_tensor * darkness_factor
    
    # Предсказания
    with torch.no_grad():
        pred_orig = torch.argmax(model(img_tensor), dim=1).item()
        pred_dark = torch.argmax(model(img_dark), dim=1).item()
        
    # Перевод индексов в исходные классы
    true_class = row['ClassId']
    class_names_true = CLASS_NAMES[true_class]
    class_names_orig = CLASS_NAMES[train_dataset.classes[pred_orig]]
    class_names_dark = CLASS_NAMES[train_dataset.classes[pred_dark]]

    # Отрисовка
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_orig.resize((128, 128)))
    axes[0].set_title(f"Оригинал\nПредсказано: {class_names_orig}\n(Верно: {class_names_true})")
    axes[0].axis('off')
    
    # Для отображения темной восстанавливаем из тензора
    dark_np = img_dark.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    dark_np = (dark_np - dark_np.min()) / (dark_np.max() - dark_np.min()) # нормализация для показа
    axes[1].imshow(dark_np)
    axes[1].set_title(f"Затемнено (x0.2)\nПредсказано: {class_names_dark}\n(Верно: {class_names_true})")
    axes[1].axis('off')
    plt.savefig("стресс-тест.png", dpi=300, bbox_inches='tight')
    plt.show()

# Найдем в валидации картинку класса "Speed limit 30"
idx_30 = val_df[val_df['ClassId'] == 1].index[0]
print("СТРЕСС-ТЕСТ НА ПЛОХОЕ ОСВЕЩЕНИЕ")
test_dark_image(best_model, val_df, idx_30, darkness_factor=0.3)

In [ ]:
# Создаем папку models
import os
os.makedirs('models', exist_ok=True)

# Сохраняем веса Baseline CNN
torch.save(baseline_model.state_dict(), 'models/baseline_cnn.pth')
print("Модель Baseline CNN сохранена в models/baseline_cnn.pth")

# Сохраняем веса ResNet
torch.save(resnet_model_v2.state_dict(), 'models/resnet18_finetuned.pth')
print("Модель ResNet18 сохранена в models/resnet18_finetuned.pth")